In [8]:
!pip install --quiet pandas numpy requests scikit-learn shap matplotlib joblib streamlit
print("✅ Installation complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 63.5 MB/s eta 0:00:00
✅ Installation complete!


In [9]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

print("=" * 60)
print("PEARLS AQI PREDICTOR - COMPLETE PIPELINE")
print("=" * 60)

# Configuration for Islamabad
CITY_NAME = "Islamabad"
CITY_LAT = 33.6844
CITY_LON = 73.0479

print(f"\n📍 City: {CITY_NAME}")
print("🔄 Step 1: Fetching historical data from Open-Meteo...")

end_date = datetime.now()
start_date = end_date - timedelta(days=90)

url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
    "latitude": CITY_LAT,
    "longitude": CITY_LON,
    "start_date": start_date.strftime("%Y-%m-%d"),
    "end_date": end_date.strftime("%Y-%m-%d"),
    "hourly": "us_aqi,pm10,pm2_5,nitrogen_dioxide,ozone",
    "timezone": "auto"
}

response = requests.get(url, params=params).json()
df = pd.DataFrame(response['hourly'])
df['timestamp'] = pd.to_datetime(df['time'])

print(f"✅ Fetched {len(df)} hours of data")

print("\n🔧 Step 2: Engineering features...")
df = df.sort_values('timestamp').reset_index(drop=True)
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['aqi_change_rate'] = df['us_aqi'].diff().fillna(0)
df['aqi_lag_1'] = df['us_aqi'].shift(1).fillna(df['us_aqi'])
df['aqi_lag_24'] = df['us_aqi'].shift(24).fillna(df['us_aqi'])
df['target_aqi_24h'] = df['us_aqi'].shift(-24)
df = df.dropna().reset_index(drop=True)

print(f"✅ Created {len(df)} training samples")

print("\n🧠 Step 3: Training Random Forest model...")
features_to_use = ['hour', 'day_of_week', 'month', 'pm10', 'pm2_5',
                   'nitrogen_dioxide', 'ozone', 'aqi_change_rate',
                   'aqi_lag_1', 'aqi_lag_24']

X = df[features_to_use]
y = df['target_aqi_24h']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"✅ Model trained!")
print(f"   RMSE: {rmse:.2f}")
print(f"   MAE: {mae:.2f}")
print(f"   R²: {r2:.3f}")

print("\n💾 Step 4: Saving model and data...")
os.makedirs('model_files', exist_ok=True)
joblib.dump(model, 'model_files/aqi_model.pkl')
df.to_csv('model_files/aqi_data.csv', index=False)

# Saved feature names for the dashboard
import json
with open('model_files/feature_names.json', 'w') as f:
    json.dump(features_to_use, f)

print("✅ Files saved to 'model_files/' folder")
print("\n" + "=" * 60)
print("🎉 PIPELINE COMPLETE!")
print("=" * 60)

PEARLS AQI PREDICTOR - COMPLETE PIPELINE

📍 City: Islamabad
🔄 Step 1: Fetching historical data from Open-Meteo...
✅ Fetched 2184 hours of data

🔧 Step 2: Engineering features...
✅ Created 2160 training samples

🧠 Step 3: Training Random Forest model...
✅ Model trained!
   RMSE: 17.28
   MAE: 13.26
   R²: 0.722

💾 Step 4: Saving model and data...
✅ Files saved to 'model_files/' folder

🎉 PIPELINE COMPLETE!


In [10]:
from google.colab import files
import shutil

# Create a zip file
shutil.make_archive('aqi_project_files', 'zip', 'model_files')

print("📥 Downloading files...")
files.download('aqi_project_files.zip')
print("✅ Download complete! Extract this zip on your computer.")

📥 Downloading files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download complete! Extract this zip on your computer.
